# Notebook 3 — Hybrid Model: Credit + Transaction → New Customers

**Project:** Open Banking Credit Risk Model  
**Author:** Chetan Prakash | MSc FinTech, University of Glasgow  

---

## Core Research Hypothesis

> *A model trained on existing customers with both credit history and transaction data can accurately predict default risk for new customers who only have transaction data — no credit bureau history required.*

## Design
| Stage | Data Used |
|-------|----------|
| **Training** | 9,000 existing customers — full credit features + 56 transaction features |
| **Inference** | 1,000 new customers — transaction features only (credit cols = 0) |

## Why Logistic Regression Outperforms XGBoost Here
At inference time, the 21 credit features are zeroed because new customers have no bureau history.  
Logistic Regression naturally ignores zero-valued features and relies on the transaction signals.  
XGBoost splits on feature thresholds — zeroed features confuse its tree-routing mechanism.  
This finding supports the **regulatory case for interpretable models** in production credit scoring.

## Expected Results
```
Baseline LR      : AUC 0.71  (bureau data only)
Open Banking LR  : AUC 0.91  (transaction data only)
Hybrid LR        : AUC 0.91  (best recall — 89% defaulters caught)
Hybrid XGBoost   : AUC 0.90  (highest precision)
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
)
import xgboost as xgb

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('Libraries loaded successfully')

## 1. Load Data & Engineered Features

In [ ]:
# If running standalone, re-run feature engineering from Notebook 2
# Otherwise load pre-saved engineered features
old_credit = pd.read_csv('../data/old_cust_credit.csv')
new_credit = pd.read_csv('../data/new_cust_credit.csv')

# Load pre-engineered transaction features (output of Notebook 2)
# If not saved, re-run the engineer_features() function from Notebook 2
old_txn_feats = pd.read_csv('../data/old_txn_features.csv')
new_txn_feats = pd.read_csv('../data/new_txn_features.csv')

print(f'Old credit   : {old_credit.shape}  | Default: {old_credit["credit_defaults"].mean():.1%}')
print(f'New credit   : {new_credit.shape}  | Default: {new_credit["credit_defaults"].mean():.1%}')
print(f'Old txn feats: {old_txn_feats.shape}')
print(f'New txn feats: {new_txn_feats.shape}')

## 2. Build Hybrid Training Set

In [ ]:
def encode_credit_full(df):
    """Encode all credit features for existing customers."""
    df = df.copy()
    cats = ['checking_status','credit_history','employment','sex','housing','savings_status']
    df = pd.get_dummies(df, columns=[c for c in cats if c in df.columns], drop_first=True)
    return df

old_credit_enc = encode_credit_full(old_credit)

# Merge credit + transaction features
train_hybrid = old_credit_enc.merge(
    old_txn_feats.drop(columns=['credit_defaults']),
    on='account_id', how='inner'
)
print(f'Hybrid training set : {train_hybrid.shape}')
print(f'Default rate        : {train_hybrid["credit_defaults"].mean():.1%}')

y_train = train_hybrid['credit_defaults']
X_train = train_hybrid.drop(columns=['account_id','credit_defaults'], errors='ignore')

txn_cols  = [c for c in X_train.columns if c in old_txn_feats.columns]
cred_cols = [c for c in X_train.columns if c not in txn_cols]
print(f'\nTotal features      : {X_train.shape[1]}')
print(f'  Credit features   : {len(cred_cols)}')
print(f'  Transaction feats : {len(txn_cols)}')

## 3. Prepare Test Set — Transaction Features Only

In [ ]:
# New customers: only transaction features available
# Credit columns are set to 0 (absent at onboarding)
X2_cols = [c for c in new_txn_feats.columns
           if c not in ['account_id','credit_defaults']]
X_test  = new_txn_feats[X2_cols].reindex(columns=X_train.columns, fill_value=0)
y_test  = new_txn_feats['credit_defaults']

print(f'Test set shape      : {X_test.shape}')
print(f'Credit cols zeroed  : {len(cred_cols)} (unavailable for new customers)')
print(f'Transaction cols    : {len(txn_cols)} (the only signal used for scoring)')

# Drop NaN rows
mask_tr = X_train.notna().all(axis=1)
X_train, y_train = X_train[mask_tr], y_train[mask_tr]
mask_te = X_test.notna().all(axis=1)
X_test, y_test = X_test[mask_te], y_test[mask_te]

X_tr,X_val,y_tr,y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr)
X_val_sc = scaler.transform(X_val)
X_te_sc  = scaler.transform(X_test)
print(f'\nTrain: {X_tr.shape} | Val: {X_val.shape} | Test (new): {X_test.shape}')

## 4. Train Hybrid Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Logistic Regression
lr = LogisticRegression(penalty='l2', solver='liblinear', max_iter=3000,
                         random_state=SEED, class_weight='balanced')
lr.fit(X_tr_sc, y_tr)
cv_lr = cross_val_score(lr, X_tr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'Hybrid LR  CV AUC: {cv_lr.mean():.4f} +/- {cv_lr.std():.4f}')

# XGBoost
sp = (y_tr==0).sum()/(y_tr==1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=sp, eval_metric='logloss',
    random_state=SEED, tree_method='hist')
xgb_model.fit(X_tr_sc, y_tr, eval_set=[(X_val_sc,y_val)], verbose=False)
cv_xgb = cross_val_score(xgb_model, X_tr_sc, y_tr, cv=cv, scoring='roc_auc')
print(f'Hybrid XGB CV AUC: {cv_xgb.mean():.4f} +/- {cv_xgb.std():.4f}')

## 5. Evaluate on New Customers

In [ ]:
lr_pred  = lr.predict(X_te_sc);          lr_prob  = lr.predict_proba(X_te_sc)[:,1]
xgb_pred = xgb_model.predict(X_te_sc);   xgb_prob = xgb_model.predict_proba(X_te_sc)[:,1]

for name, pred, prob in [('Hybrid Logistic Regression', lr_pred, lr_prob),
                          ('Hybrid XGBoost',            xgb_pred, xgb_prob)]:
    print(f'\n--- {name} ---')
    print(f'  AUC      : {roc_auc_score(y_test, prob):.4f}')
    print(f'  Accuracy : {accuracy_score(y_test, pred):.4f}')
    print(f'  Brier    : {brier_score_loss(y_test, prob):.4f}')
    cm = confusion_matrix(y_test, pred)
    fn, tp = cm[1,0], cm[1,1]
    print(f'  Defaulters caught: {tp}/{fn+tp} ({tp/(fn+tp)*100:.0f}%)')
    print(classification_report(y_test, pred, target_names=['No Default','Default']))

## 6. Visualisations

In [ ]:
# ROC + Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax = axes[0]
for prob, label, color in [(lr_prob,'Hybrid LR','steelblue'),
                             (xgb_prob,'Hybrid XGB','purple')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f'{label} (AUC={roc_auc_score(y_test,prob):.3f})')
ax.plot([0,1],[0,1],'--',color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC — Hybrid Model\n(New Customers, Txn Only)', fontweight='bold')
ax.legend(loc='lower right')

for ax, pred, title, cmap in zip(axes[1:],
    [lr_pred, xgb_pred],['Hybrid LR','Hybrid XGB'],['Blues','Purples']):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
        display_labels=['No Default','Default']).plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f'{title}\nNew Customers', fontweight='bold')

plt.suptitle('Notebook 3 — Hybrid Model (New Customers Scored on Transaction Data Only)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/03_hybrid_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance — credit vs transaction
importances = pd.Series(xgb_model.feature_importances_, index=X_tr.columns)
top20 = importances.nlargest(20).sort_values()
colors = ['teal' if c in txn_cols else 'steelblue' for c in top20.index]

plt.figure(figsize=(11, 8))
top20.plot(kind='barh', color=colors, edgecolor='white')
legend_elements = [
    mpatches.Patch(facecolor='teal',      label='Transaction Feature'),
    mpatches.Patch(facecolor='steelblue', label='Credit Feature')
]
plt.legend(handles=legend_elements, loc='lower right')
plt.title('Hybrid XGBoost — Top 20 Features\n(teal=Transaction | blue=Credit)',
          fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/03_feature_importance_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

txn_in_top20 = sum(1 for c in top20.index if c in txn_cols)
print(f'Top 20 features: {txn_in_top20} transaction-based, {20-txn_in_top20} credit-based')

In [ ]:
# Full model comparison chart
comparison = pd.DataFrame([
    {'Model':'Baseline — LR',       'Type':'Traditional',  'AUC':0.7103,'Brier':0.1942,'Recall':43.2},
    {'Model':'Baseline — XGB',      'Type':'Traditional',  'AUC':0.6804,'Brier':0.2042,'Recall':32.7},
    {'Model':'Open Banking — LR',   'Type':'Transaction',  'AUC':0.9112,'Brier':0.1166,'Recall':84.3},
    {'Model':'Open Banking — XGB',  'Type':'Transaction',  'AUC':0.8992,'Brier':0.1218,'Recall':73.8},
    {'Model':'Hybrid — LR',         'Type':'Hybrid',       'AUC':roc_auc_score(y_test,lr_prob),
                                     'Brier':brier_score_loss(y_test,lr_prob),
                                     'Recall':sum(lr_pred[y_test==1])/sum(y_test==1)*100},
    {'Model':'Hybrid — XGB',        'Type':'Hybrid',       'AUC':roc_auc_score(y_test,xgb_prob),
                                     'Brier':brier_score_loss(y_test,xgb_prob),
                                     'Recall':sum(xgb_pred[y_test==1])/sum(y_test==1)*100},
])

color_map = {'Traditional':'steelblue','Transaction':'teal','Hybrid':'purple'}
bar_colors = [color_map[t] for t in comparison['Type']]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric, xlabel in zip(axes,
    ['AUC','Brier','Recall'],
    ['AUC Score (higher = better)','Brier Score (lower = better)','Recall / Defaulters Caught (%)']):
    bars = ax.barh(comparison['Model'], comparison[metric],
                   color=bar_colors, edgecolor='white', height=0.6)
    for bar, val in zip(bars, comparison[metric]):
        ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                f'{val:.3f}' if metric!='Recall' else f'{val:.1f}%',
                va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_title(metric, fontweight='bold')
    if metric == 'AUC':
        ax.axvline(0.5, color='red', linestyle='--', alpha=0.4, label='Random')
        ax.set_xlim(0.5, 1.05)

legend_els = [mpatches.Patch(facecolor=c, label=l)
              for l, c in color_map.items()]
fig.legend(handles=legend_els, loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.05))
plt.suptitle('Project Conclusion: Open Banking Data Delivers +28% AUC Improvement\nover Traditional Bureau-Only Baseline',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Results

| Model | AUC | Brier | Accuracy | Defaulters Caught |
|-------|-----|-------|----------|-------------------|
| Baseline — Logistic Regression | 0.7103 | 0.1942 | 69.9% | 43% |
| Baseline — XGBoost | 0.6804 | 0.2042 | 67.4% | 33% |
| Open Banking — Logistic Regression | 0.9112 | 0.1166 | 84.4% | 84% |
| Open Banking — XGBoost | 0.8992 | 0.1218 | 83.0% | 74% |
| **Hybrid — Logistic Regression** | **0.9108** | **0.1407** | **80.2%** | **89%** |
| Hybrid — XGBoost | 0.8997 | 0.1214 | 83.3% | 78% |

## Conclusion

**Hypothesis confirmed.** A hybrid model trained on existing customers (credit + transactions) achieves **AUC 0.91** when scoring new customers using only 90 days of transaction data — catching **89% of defaulters** while missing just 35 out of 324.

This represents a **+28% AUC improvement** over the traditional bureau-only baseline.

**Key finding on model selection:** Logistic Regression outperforms XGBoost in the hybrid setting because zeroed credit features at inference time disrupt XGBoost's tree-splitting mechanism. This supports the **regulatory preference for interpretable models** in credit scoring — the transparent model is not just explainable, it performs better in the realistic thin-file deployment scenario.